In [ ]:
import sys
import os

# Get the absolute path to the root directory where UQ_toolbox.py is located
root_dir = os.path.abspath(os.path.join(os.path.dirname('medMNIST'), '..'))
sys.path.append(root_dir)

In [ ]:
import torch
import torchvision.transforms as transforms
from medMNIST.utils import train_load_datasets_resnet as tr
import numpy as np
from torch.utils.data import Subset

In [ ]:
def get_labels_array(ds):
    """
    Return a 1D numpy array of labels for any torch Dataset, possibly nested in Subset(s).
    Handles datasets exposing `.labels` (e.g., MedMNIST) or `.targets` (e.g., torchvision).
    As a last resort, indexes ds[i][1] for all i (slow, but generic).
    """
    # Unwrap nested Subset(s) and compose indices
    idx = None
    while isinstance(ds, Subset):
        if idx is None:
            idx = np.array(ds.indices)
        else:
            # compose indices if we have Subset(Subset(base))
            idx = np.array(ds.indices)[idx]
        ds = ds.dataset

    # Now ds is the base dataset
    if hasattr(ds, "labels"):
        labels = np.array(ds.labels)
    elif hasattr(ds, "targets"):
        labels = np.array(ds.targets)
    else:
        # Generic (slow) fallback: index the dataset to get labels
        labels = np.array([ds[i][1] for i in range(len(ds))])

    # Squeeze to 1D in case of shape (N,1)
    labels = np.asarray(labels).squeeze()

    # Apply subset indices if any
    if idx is not None:
        labels = labels[idx]

    return labels

def get_class_counts(ds):
    labels_arr = get_labels_array(ds)
    _, counts = np.unique(labels_arr, return_counts=True)
    return counts


In [ ]:
def evenness_stats(counts):
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    S = len(counts)
    if total == 0 or S == 0:
        return {"S": S, "J": 0.0, "Neff": 0.0, "Neff_over_S": 0.0,
                "TV": 0.0, "IR": np.nan, "p_max": 0.0, "p_min": 0.0}
    p = counts / total
    p_nz = p[p > 0]
    H = -(p_nz * np.log(p_nz)).sum()
    J = H / np.log(S) if S > 1 else 1.0
    Neff = float(np.exp(H))
    TV = 0.5 * np.abs(p - 1.0 / S).sum()
    p_min = float(p_nz.min()) if p_nz.size else 0.0
    IR = float(p.max() / p_min) if p_min > 0 else np.inf
    return {
        "S": S,
        "J": float(J),
        "Neff": Neff,
        "Neff_over_S": float(Neff / S),
        "TV": float(TV),
        "IR": IR,
        "p_max": float(p.max()),
        "p_min": p_min,
    }

def format_stats(st):
    S = int(round(st['S']))
    return (f"J={st['J']:.3f}, Neff={st['Neff']:.2f}/{S} ({st['Neff_over_S']:.1%}), "
            f"TV={st['TV']:.3f}, IR={st['IR']:.2f}, pmin={st['p_min']:.1%}, pmax={st['p_max']:.1%}")


def aggregate_over_loaders(loaders, get_class_counts_fn):
    sizes = [len(ld.dataset) for ld in loaders]
    stats_list = [evenness_stats(get_class_counts_fn(ld.dataset)) for ld in loaders]
    keys = ["J","Neff","Neff_over_S","TV","IR","p_max","p_min","S"]
    mean_stats = {k: float(np.mean([d[k] for d in stats_list])) for k in keys}
    mean_size = int(round(np.mean(sizes)))
    return mean_size, mean_stats

In [ ]:
flags = ['breastmnist', 'organamnist', 'pneumoniamnist', 'dermamnist', 'octmnist', 'pathmnist', 'bloodmnist', 'tissuemnist']
colors = [False, False, False, True, False, True, True, False]  # Colors for the flags
device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
size = 224  # Image size for the models
batch_size = 4000  # Batch size for the DataLoader
model_global_perfs = {}

for flag, color in zip(flags, colors):
    if color is True:
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[.5, .5, .5], std=[.5, .5, .5])
        ])
    else:
        # For grayscale images, repeat the single channel to make it compatible with ResNet
        # ResNet expects 3 channels, so we repeat the single channel image
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[.5], std=[.5]),
            transforms.Lambda(lambda x: x.repeat(3, 1, 1))
        ])
    
    [train_dataset, calibration_dataset, test_dataset], [train_loader, calibration_loader, test_loader], info = tr.load_datasets(flag, color, size, transform, batch_size)
    train_loaders, val_loaders = tr.CV_train_val_loaders(None, train_dataset, batch_size=batch_size)
    print(f"{flag}:")


   
    # Train / Val (aggregate across folds)
    mean_train_size, train_stats = aggregate_over_loaders(train_loaders, get_class_counts)
    mean_val_size,   val_stats   = aggregate_over_loaders(val_loaders,   get_class_counts)



    calib_size = len(calibration_dataset)
    test_size = len(test_dataset)

    train_equitabilities = [evenness_stats(get_class_counts(loader.dataset)) for loader in train_loaders]
    val_equitabilities = [evenness_stats(get_class_counts(loader.dataset)) for loader in val_loaders]


    # Calibration set
    calib_labels = get_labels_array(calibration_dataset)
    calib_class_counts = np.unique(calib_labels, return_counts=True)[1]

    # test_dataset is a base medmnist dataset (not a Subset)
    test_labels = np.array(test_dataset.labels).flatten()
    test_class_counts = np.unique(test_labels, return_counts=True)[1]

    # Calibration / Test (single dataset each)
    calib_stats = evenness_stats(calib_class_counts)
    test_stats  = evenness_stats(test_class_counts)

    print(f"  Train  mean N={mean_train_size} ({len(train_loaders)} folds): {format_stats(train_stats)}")
    print(f"  Val    mean N={mean_val_size} ({len(val_loaders)} folds):   {format_stats(val_stats)}")
    print(f"  Calibration N={calib_size}: {format_stats(calib_stats)}")
    print(f"  Test         N={test_size}: {format_stats(test_stats)}")